### 크롤링으로 웹 페이지 정보를 수집하기 전 주의사항
* `robots.txt` 파일 확인해서 수집을 허용하는지 확인
* 이용약관에서 자동 수집 금지, 상업적 이용 금지, 재배포 금지 조항을 확인
* 요청 간격을 적절히 지정

### 크롤링 과정
1. requests로 화면 html 코드를 가져옴
2. BS로 원하는 요소를 탐색
3. 탐색한 요소에 있는 정보를 추출
4. 활용(DataFrame)

### 1.requests로 화면 html 코드를 가져옴

In [103]:
import requests as req

BASE_URL = "https://kin.naver.com"


def fetch_kin_html(query:str, page:int=1) -> str | None:
    try:
        # params의 역할(get) : url 뒤에 ? 변수=값 형태로 변환
        # query=값&page=값
        result = req.get(f"{BASE_URL}/search/list.naver", params={"query":query, "page":page}, timeout=5)
        print(result.url)
        result.raise_for_status()
        return result.text
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None

html = fetch_kin_html("파이썬", 1)
# print(html.strip()) # 확인용

https://kin.naver.com/search/list.naver?query=%ED%8C%8C%EC%9D%B4%EC%8D%AC&page=1


### 2. BS로 원하는 요소를 탐색
* 탐색하기 전 원하는 요소의 선택자를 확인
* soup = BeautifulSoup(html코드, "html.parser") : html코드를 html객체로 분석
* els = soup.select("선택자") : 선택자와 일치한느 모든 요소들. 리스트로 반환
* el = soup.select_one("선택자") : 선택자와 일치하는 첫번째 요소.
* el.get_text(strip=True) : 공백을 제거하고 텍스트를 가져옴
* el.get("속성명"), el["속성명"] : 요소의 속성명과 일치하는 속성의 값을 가져옴

In [104]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

links = soup.select("._searchListTitleAnchor")
len(links)

10

### 3. 탐색한 요소에 있는 정보를 추출

In [105]:
list = []
for link in links:
    # 제목
    title = link.get_text(strip=True)
    # 제목 링크
    href = link.get("href")
    list.append({"title" : title, "href" : href})

### 4.활용(DataFrame)

In [106]:
import pandas as pd
df = pd.DataFrame(list)
df.head(5)

,title,href
0,정보 과목파이썬코드 질문,https://kin.naver.com/qna/detail.naver?dirId=1...
1,파이썬아이콘만드는방법,https://kin.naver.com/qna/detail.naver?dirId=1...
2,"파이썬기초 질문(urllib, beautifulsoup4)",https://kin.naver.com/qna/detail.naver?dirId=1...
3,파이썬리스트 슬라이싱,https://kin.naver.com/qna/detail.naver?dirId=1...
4,파이썬으로 구현하는 블로그 자동 포스팅 기....,https://kin.naver.com/qna/detail.naver?dirId=1...


### 응용
* 검색어("파이썬")으로 등록된 질문과 질문 상세, 첫번째 답변을 크롤링하는 작업
* df에 있는 href를 이용하여 질문 상세와 대표답변 1개를 가져오는 작업
* 결과물은 df에 title, title_detail, answer가 들어가도록 작업

In [107]:
# 질문 링크를 이용하여 질문 상세 페이지 html 코드 가져오기
href = df.iloc[0]["href"]

# href를 이용해서 질문 상세 페이지에 있는 html 코드 가져오기
result = req.get(href, timeout=5)
html = result.text

In [108]:
# bs를 이용해서 요소를 탐색(질문 상세, 첫번째 답변)
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")
detail_el = soup.select_one(".questionDetail")
answer_el = soup.select_one(".answerDetail")

In [109]:
# 요소에서 필요한 내용 추출(질문 상세, 첫번째 답변)
detail = detail_el.get_text(strip=True)
answer = answer_el.get_text(strip=True) if answer_el is not None else None # 답변이 등록 안된 경우는 None으로
# print(detail)
# print(answer)

In [110]:
import time

def fetch_kin_detail_html(href:str)->str|None:
    try:
        result = req.get(href, timeout=5)
        result.raise_for_status()
        return result.text
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None

def get_detail_answer(html:str)->dict|None:
    if html is None:
        return None
    soup = BeautifulSoup(html, "html.parser")
    detail_el = soup.select_one(".questionDetail")
    answer_el = soup.select_one(".answerDetail")
    detail = detail_el.get_text(strip=True)
    answer = answer_el.get_text(strip=True) if answer_el is not None else None
    return {"detail" : detail, "answer" : answer}
    
list = []
for row in df.itertuples():
    href = row.href
    html = fetch_kin_detail_html(href)
    result = get_detail_answer(html)
    list.append(result)
    time.sleep(0.1)



In [111]:
# list를 DataFrame으로 변환
df_answer = pd.DataFrame(list)
df_answer.head(3)

,detail,answer
0,이 두문제 풀이랑 답좀 알려주세요,두 문제 모두 차근차근 따라가 볼게요.​1. 16번 (출력되는 줄 수)​print(...
1,아이콘 어떻게만드나요?,해결 방법팟플레이어 오디오 트랙 변경팟플레이어로 영상 재생 중 Alt + A를 누르...
2,이제 막 파이썬 기초 배우기 시작했는데urllib 명령어를 넣으면 이런 오류가 뜹니...,에러 메시지를 끝까지 보면 코드가 틀린 게 아니라 실행 환경의 인터넷이 막혀 있는 ...


In [112]:
# 기존 df와 답변이 있는 df_answer를 합침
df_merged = pd.merge(df, df_answer, left_index=True, right_index=True)
df_merged.head(3)

,title,href,detail,answer
0,정보 과목파이썬코드 질문,https://kin.naver.com/qna/detail.naver?dirId=1...,이 두문제 풀이랑 답좀 알려주세요,두 문제 모두 차근차근 따라가 볼게요.​1. 16번 (출력되는 줄 수)​print(...
1,파이썬아이콘만드는방법,https://kin.naver.com/qna/detail.naver?dirId=1...,아이콘 어떻게만드나요?,해결 방법팟플레이어 오디오 트랙 변경팟플레이어로 영상 재생 중 Alt + A를 누르...
2,"파이썬기초 질문(urllib, beautifulsoup4)",https://kin.naver.com/qna/detail.naver?dirId=1...,이제 막 파이썬 기초 배우기 시작했는데urllib 명령어를 넣으면 이런 오류가 뜹니...,에러 메시지를 끝까지 보면 코드가 틀린 게 아니라 실행 환경의 인터넷이 막혀 있는 ...


## 최종 결과물
### fetch_kin_html

In [113]:
import requests as req

def fetch_kin_html(query:str, page:int=1) -> str | None:
    BASE_URL = "https://kin.naver.com"
    try:
        # params의 역할(get) : url 뒤에 ? 변수=값 형태로 변환
        # query=값&page=값
        result = req.get(f"{BASE_URL}/search/list.naver", params={"query":query, "page":page}, timeout=5)
        print(result.url)
        result.raise_for_status()
        return result.text
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None

In [114]:
from bs4 import BeautifulSoup
def get_question_list(html:str | None)->list[dict]|None:
    if html is None:
        return None
    soup = BeautifulSoup(html, "html.parser")
    links = soup.select("._searchListTitleAnchor")
    if links is None:
        return None
    list = []
    for link in links:
        # 제목
        title = link.get_text(strip=True)
        # 제목 링크
        href = link.get("href")
        list.append({"title" : title, "href" : href})
    return list



In [115]:
def get_question_df(list:list[dict]|None)->pd.DataFrame|None:
    if list is None:
        return None
    return pd.DataFrame(list)
    

In [116]:
import time

def fetch_kin_detail_html(href:str)->str|None:
    try:
        result = req.get(href, timeout=5)
        result.raise_for_status() # 정상응답이 아니면 예외를 발생시킴
        return result.text # 응답 결과에서 문자들을 반환(html 코드)
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None

def get_detail_answer(html:str|None)->dict|None:
    if html is None:
        return None
    soup = BeautifulSoup(html, "html.parser") #html코드로 된 문자열을 탐색이 쉽게 만들어줌
    # 원하는 요소를 탐색
    detail_el = soup.select_one(".questionDetail")
    answer_el = soup.select_one(".answerDetail")
    detail = detail_el.get_text(strip=True)
    answer = answer_el.get_text(strip=True) if answer_el is not None else None # 답변이 등록 안된 경우는 None으로 
    return { "detail" : detail, "answer" : answer}

In [117]:
def get_answer_df(df:pd.DataFrame | None)->pd.DataFrame|None:
    if df is None:
        return None
    list = []    
    for row in df.itertuples():
        href = row.href
        html = fetch_kin_detail_html(href)
        result = get_detail_answer(html)
        list.append(result)
        time.sleep(0.1) # 너무 빠른 시간에 많이 요청하면 ip차단될수 있어서 지연요청
    return pd.DataFrame(list)

In [118]:
def crawling_kin(query:str, max_page:int = 1)->pd.DataFrame:
    if max_page <= 0:
        max_page = 1
    df_total = pd.DataFrame(columns=["title", "href", "detail", "answer"])
    for page in range(1, max_page+1):
        html = fetch_kin_html(query, page)
        list = get_question_list(html)
        df = get_question_df(list)
        df_answer = get_answer_df(df)
        df_merged = pd.merge(df, df_answer, left_index=True, right_index=True)
        df_total = pd.concat([df_total, df_merged], ignore_index=True)
        print(f"{page}/{max_page} 크롤링 완료")
    return df_total

In [119]:
df = crawling_kin("파이썬", 2)
print(df.shape)
# df

https://kin.naver.com/search/list.naver?query=%ED%8C%8C%EC%9D%B4%EC%8D%AC&page=1
1/2 크롤링 완료
https://kin.naver.com/search/list.naver?query=%ED%8C%8C%EC%9D%B4%EC%8D%AC&page=2
2/2 크롤링 완료
(20, 4)
